In [112]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from transformers import AutoTokenizer, AutoModel


Load Dataset

In [113]:
df = pd.read_csv("ats_dataset.csv")


In [114]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14546 entries, 0 to 14545
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   job_description  14546 non-null  object 
 1   resume           14546 non-null  object 
 2   score            14546 non-null  float64
 3   type             14546 non-null  object 
dtypes: float64(1), object(3)
memory usage: 454.7+ KB


In [95]:


df = df[["job_description", "resume", "score"]]

df.head()

,job_description,resume,score
0,We are looking for a talented Frontend Develo...,"As a seasoned Frontend Developer, I have a pro...",0.9
1,We are seeking an experienced Backend Develope...,With a solid background in Backend Development...,0.9
2,We are hiring a Data Scientist who is passiona...,"With a background in Data Science, I possess a...",0.9
3,We are looking for a talented Frontend Develo...,Experienced Frontend Developer with a passion ...,0.9
4,We are looking for a talented Frontend Develo...,Passionate Frontend Developer with over 4 year...,0.9


Build Vocabulary

In [115]:
from collections import Counter

def build_vocab(texts, min_freq=2):

    counter = Counter()

    for text in texts:
        counter.update(text.lower().split())

    vocab = {
        word: idx + 1
        for idx, (word, freq) in enumerate(counter.items())
        if freq >= min_freq
    }

    vocab["<PAD>"] = 0

    return vocab

Create Vocabulary

In [97]:
all_texts = (
    df["job_description"].tolist() +
    df["resume"].tolist()
)

vocab = build_vocab(all_texts)

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 1326


Text Encoding

In [98]:
MAX_LEN = 100

def encode_text(text):

    tokens = text.lower().split()

    ids = [
        vocab.get(token, 0)
        for token in tokens
    ]

    ids = ids[:MAX_LEN]

    padding = [0] * (MAX_LEN - len(ids))

    ids += padding

    return torch.tensor(ids)

Train/Test Split

In [99]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

Create Dataset Class

In [100]:
class ResumeDataset(Dataset):

    def __init__(self, dataframe):

        self.df = dataframe.reset_index(drop=True)

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        jd = str(row["job_description"])

        resume = str(row["resume"])

        score = float(row["score"])

        return jd, resume, score    

Tokenizer

In [116]:
tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

Collate Function

In [117]:
def collate_fn(batch):

    jds = []
    resumes = []
    scores = []

    for jd, resume, score in batch:

        jds.append(str(jd))

        resumes.append(str(resume))

        scores.append(float(score))

    # tokenize JD
    jd_inputs = tokenizer(

        jds,

        padding=True,

        truncation=True,

        return_tensors="pt",

        max_length=64
    )

    # tokenize resumes
    resume_inputs = tokenizer(

        resumes,

        padding=True,

        truncation=True,

        return_tensors="pt",

        max_length=64
    )

    # scores tensor
    scores = torch.tensor(

        scores,

        dtype=torch.float
    )

    return jd_inputs, resume_inputs, scores

DataLoaders

In [122]:
train_loader = DataLoader(

    train_dataset,

    batch_size=16,

    shuffle=True,

    collate_fn=collate_fn
)

test_loader = DataLoader(

    test_dataset,

    batch_size=16,

    shuffle=False,

    collate_fn=collate_fn
)

BERT Similarity Model

In [104]:
from transformers import AutoModel

class SimilarityModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.bert = AutoModel.from_pretrained(
            "bert-base-uncased"
        )

        for param in self.bert.parameters():
            param.requires_grad = False

        self.fc = nn.Sequential(

            nn.Linear(768 * 3, 256),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(256, 1),

            nn.Sigmoid()
        )

    def forward(
        self,
        jd_inputs,
        resume_inputs
    ):

        jd_vec = self.bert(
            **jd_inputs
        ).pooler_output

        resume_vec = self.bert(
            **resume_inputs
        ).pooler_output

        diff = torch.abs(
            jd_vec - resume_vec
        )

        combined = torch.cat(
            [jd_vec, resume_vec, diff],
            dim=1
        )

        score = self.fc(combined)

        return score.squeeze()

Initialize Model

In [ ]:
device = torch.device("cpu")

model = SimilarityModel().to(device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5736.58it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loss + Optimizer

In [106]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

Training Loop

In [107]:
EPOCHS = 3
for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for jd_inputs, resume_inputs, score in train_loader:

        jd_inputs = {
            k: v.to(device)
            for k, v in jd_inputs.items()
        }

        resume_inputs = {
            k: v.to(device)
            for k, v in resume_inputs.items()
        }

        score = score.to(device)

        preds = model(
            jd_inputs,
            resume_inputs
        )

        loss = criterion(preds, score)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}, Loss: {total_loss:.4f}"
    )

Epoch 1, Loss: 5.1401
Epoch 2, Loss: 4.5206
Epoch 3, Loss: 4.4039


Evaluate Accuracy

In [124]:
model.eval()

predictions = []

actuals = []

with torch.no_grad():

    for jd_inputs, resume_inputs, scores in test_loader:

        # move JD tensors
        jd_inputs = {

            k: v.to(device)

            for k, v in jd_inputs.items()
        }

        # move resume tensors
        resume_inputs = {

            k: v.to(device)

            for k, v in resume_inputs.items()
        }

        scores = scores.to(device)

        # predictions
        preds = model(

            jd_inputs,

            resume_inputs
        )

        predictions.extend(

            preds.cpu().numpy()
        )

        actuals.extend(

            scores.cpu().numpy()
        )

In [125]:
rmse = np.sqrt(

    mean_squared_error(
        actuals,
        predictions
    )
)

print("RMSE:", rmse)

RMSE: 0.08157781771222945


Accuracy Score

In [126]:
binary_preds = [
    1 if p >= 0.5 else 0
    for p in predictions
]

binary_actuals = [
    1 if a >= 0.5 else 0
    for a in actuals
]

Compute Accuracy

In [127]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(

    binary_actuals,

    binary_preds
)

print(

    "Accuracy:",

    round(acc * 100, 2),

    "%"
)

Accuracy: 98.08 %


save

In [ ]:
torch.save(

    model.state_dict(),

    "resume_model.pth"
)

print("Model Saved")

Model Saved
